# 🧠 PyTorch Neural Network - MNIST Digit Classification

This notebook demonstrates building, training, and evaluating a Multi-Layer Perceptron (MLP) neural network using **PyTorch** on the classic **MNIST** dataset of handwritten digits.

## 1. Imports and Environment Setup

We import PyTorch modules for building model layers (`torch.nn`), handling dataset transformations (`torchvision.transforms`), loading datasets (`torchvision.datasets`), and batching (`DataLoader`, `random_split`).

In [1]:
from torchvision import datasets, transforms
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split

## 2. Define Data Transformations

We construct a preprocessing pipeline:
1. `ToTensor()`: Converts PIL images to PyTorch tensors and scales pixel values from `[0, 255]` to `[0.0, 1.0]`.
2. `Normalize((0.1307,), (0.3081,))`: Standardizes pixel values using the global mean (`0.1307`) and standard deviation (`0.3081`) of the MNIST dataset.
3. `Lambda(lambda x: x.view(-1))`: Flattens 2D `28x28` image tensors into a 1D vector of length `784`.

In [2]:
# Define preprocessing transforms for training and test data
transform = transforms.Compose([
    transforms.ToTensor(),                           # Scale pixels to [0.0, 1.0]
    transforms.Normalize((0.1307,), (0.3081,)),     # Standardize with MNIST mean & std
    transforms.Lambda(lambda x: x.view(-1))         # Flatten 28x28 image to 784 vector
])

## 3. Download and Load MNIST Dataset

We load the official MNIST dataset for training (`60,000` samples) and testing (`10,000` samples), applying our pre-defined transformation pipeline.

In [3]:
# Download and load MNIST train and test sets
train_ds = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_ds = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

## 4. Train-Validation Split

We split the 60,000 training images into **55,000 training samples** and **5,000 validation samples**. A seed is set for reproducibility.

In [4]:
# Reproducible split into 55,000 train and 5,000 validation samples
generator = torch.Generator().manual_seed(42)
train, valid = random_split(train_ds, [55000, 5000], generator=generator)

## 5. Create DataLoaders

DataLoaders partition the datasets into mini-batches (`batch_size=128`). We set `shuffle=True` for training to randomize sample ordering across epochs.

In [5]:
# Initialize DataLoaders for batch processing
trainloader = DataLoader(train, batch_size=128, shuffle=True)
validloader = DataLoader(valid, batch_size=128, shuffle=False)
testloader = DataLoader(test_ds, batch_size=128, shuffle=False)

## 6. Define Neural Network Architecture

We build a 3-layer fully connected Neural Network (MLP):
* **Input Layer**: `784` features → `128` neurons (`ReLU` activation)
* **Hidden Layer**: `128` neurons → `64` neurons (`ReLU` activation)
* **Output Layer**: `64` neurons → `10` logits (one per class)

In [6]:
# Set accelerator hardware (GPU if available, else CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Build sequential fully-connected architecture
model = nn.Sequential(
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 10)
).to(device)

## 7. Define Loss Function and Optimizer

* **Loss Function**: `nn.CrossEntropyLoss()` combines `LogSoftmax` and negative log likelihood loss (`NLLLoss`), expecting unnormalized logits directly.
* **Optimizer**: `Adam` with a learning rate of `0.001` (`1e-3`).

In [7]:
# Configure loss function and Adam optimizer
criterion = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

## 8. Train and Validate the Model

We train the model for 15 epochs. In each epoch, we:
1. Perform forward and backward passes on training batches to update network weights.
2. Evaluate performance on validation data in `torch.no_grad()` mode.

In [8]:
epochs = 15

for epoch in range(epochs):
    # --- TRAINING PHASE ---
    model.train()
    running_loss = 0.0
    correct_train = 0.0
    total_train = 0.0

    for inputs, targets in trainloader:
        inputs, targets = inputs.to(device), targets.to(device)
        opt.zero_grad()
        output = model(inputs)
        loss = criterion(output, targets)
        loss.backward()
        opt.step()
        
        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(output, 1)
        correct_train += (preds == targets).sum().item()
        total_train += targets.size(0)

    train_loss = running_loss / total_train
    train_acc = correct_train / total_train

    # --- VALIDATION PHASE ---
    model.eval()
    val_loss = 0.0
    correct_val = 0.0
    total_val = 0.0

    with torch.no_grad():
        for inputs, targets in validloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            val_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct_val += (preds == targets).sum().item()
            total_val += targets.size(0)

    val_loss = val_loss / total_val
    val_acc = correct_val / total_val

    print(f"Epoch {epoch+1:02d}/{epochs} | "
          f"Train Loss: {train_loss:.4f} - Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} - Val Acc: {val_acc:.4f}")

## 9. Final Test Evaluation

Finally, we evaluate the trained model on the unseen test set (`10,000` images) to obtain final accuracy.

In [9]:
# Evaluate trained model on test dataset
model.eval()
correct_test = 0
total_test = 0

with torch.no_grad():
    for inputs, targets in testloader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        correct_test += (preds == targets).sum().item()
        total_test += targets.size(0)

print(f"\nFinal Test Accuracy: {correct_test / total_test:.4f}")